# NumPy: la peça que us falta per fer machine learning

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Ja sabeu Python bàsic (tipus, variables, condicionals, llistes, bucles) i ja heu vist,
conceptualment, com un model de machine learning classifica flors d'iris mirant les
seves mesures. Us falta una peça per poder fer-ho vosaltres mateixos: **com es
manipulen taules de números de manera eficient**. Aquesta peça es diu **NumPy**.

En aquest quadern no entrenareu cap model encara. El proper pas, quan ja sapigueu
manejar l'array, és fer-ho de veritat amb els exercicis de machine learning que teniu
preparats. Sense NumPy, no es pot.

## Els objectes d'aquest quadern

Tot el que fareu en aquest quadern passa per dos objectes. Aquesta taula és la
referència que heu de tenir a la vista mentre feu els exercicis: els noms dels
mètodes no s'han d'endevinar, són aquests.

### `ndarray` — l'array de NumPy

És el que torna `np.array(...)`, `np.zeros(...)`, `np.arange(...)`, `iris.data`, i
també qualsevol llesca (`X[:, 2]`) o resultat d'una operació (`a + b`).

| Atribut (sense parèntesis) | Què val |
| --- | --- |
| `.shape` | la forma, com a tupla: `(150, 4)` vol dir 150 files i 4 columnes |
| `.dtype` | el tipus de dada que guarda: `float64`, `int64`, `bool`... |
| `.ndim` | en quantes dimensions està organitzat: 1 és un vector, 2 una taula |
| `.size` | el nombre total d'elements |
| `.T` | el mateix array transposat: files i columnes intercanviades |

| Mètode (amb parèntesis) | Què fa |
| --- | --- |
| `.mean()` | la mitjana. Amb `axis=0`, una per columna; amb `axis=1`, una per fila |
| `.std()` | la desviació típica, amb el mateix `axis` |
| `.min()` / `.max()` | el valor mínim / màxim |
| `.sum()` | la suma. Sobre una màscara booleana, compta els `True` |
| `.argmax()` / `.argmin()` | la **posició** del màxim / mínim, no el valor |
| `.reshape(files, columnes)` | el mateix contingut reorganitzat amb una forma nova |
| `.copy()` | una còpia independent, que una llesca no és (punt 3) |
| `.astype(tipus)` | el mateix contingut convertit a un altre `dtype` |
| `.round(decimals)` | els valors arrodonits |

### `Bunch` — el que torna `load_iris()`

No és un array: és el contenidor que porta el dataset, les etiquetes i els noms.

| Atribut | Què hi ha a dins |
| --- | --- |
| `.data` | l'`ndarray` de les mesures: 150 files × 4 columnes |
| `.target` | l'`ndarray` amb l'espècie de cada flor, com a número `0`/`1`/`2` |
| `.feature_names` | la llista amb els noms de les 4 columnes |
| `.target_names` | la llista amb els noms de les 3 espècies |

Fixeu-vos que `.data` i `.target` són atributs del `Bunch`, i que cadascun d'ells és
un `ndarray`: per tant tenen, al seu torn, tot el de la taula de dalt.
`iris.data.shape` és legítim, i val `(150, 4)`.

### Atribut o mètode: la distinció que més costa

Un **atribut** és un valor que l'objecte ja porta a dins, i es llegeix **sense
parèntesis**. Un **mètode** és un càlcul que l'objecte sap fer, i s'invoca **amb
parèntesis**.

```python
X.shape      # atribut: (150, 4)
X.shape()    # TypeError: 'tuple' object is not callable
X.mean()     # metode: el numero que buscaveu
X.mean       # sense parentesis no calcula res: imprimeix el metode mateix
```

Confondre-ho no dona un número equivocat, dona un `TypeError` o un objecte estrany a
pantalla. El quadern `FO_00_objectes_i_autocompletar.ipynb` ho tracta a fons.

Dues coses que podeu fer sempre, i també a Colab:

- escriure `objecte.` i prémer **TAB**: surt la llista de tot el que aquell objecte té.
- escriure `objecte.metode?` i executar la cel·la: surt la documentació del mètode,
  amb els arguments que accepta.

## 1. Per què NumPy i no llistes

Ja sabeu sumar números amb una llista de Python i un bucle. Funciona. El problema
apareix quan les dades es fan grans: un dataset real pot tenir milers o milions de
files. Provem-ho amb un milió de números: sumar el doble de cadascun, primer amb una
llista i un bucle, després amb un array de NumPy.

In [ ]:
import time
import numpy as np

n = 1_000_000
llista = list(range(n))
array = np.arange(n)

inici = time.perf_counter()
resultat_llista = [x * 2 for x in llista]
temps_llista = time.perf_counter() - inici

inici = time.perf_counter()
resultat_array = array * 2
temps_array = time.perf_counter() - inici

print(f"Amb llista de Python: {temps_llista * 1000:.1f} ms")
print(f"Amb array de NumPy:   {temps_array * 1000:.1f} ms")
print(f"NumPy és {temps_llista / temps_array:.0f} vegades més ràpid")

No és un error d'arrodoniment: NumPy ha sortit unes **25 vegades més ràpid**, amb
una sola operació. Amb operacions encadenades (sumar, elevar al quadrat, treure
l'arrel..., que és exactament el que fareu per calcular distàncies) la diferència es
multiplica.

Per què passa això? Dues raons:

- Una llista de Python guarda **objectes** de qualsevol tipus, dispersos per la
  memòria; per multiplicar cada element, Python ha de mirar de quin tipus és cada
  vegada. És el preu de la flexibilitat.
- Un array de NumPy guarda **un sol tipus** de dada, en un bloc de memòria
  **contigu**. Multiplicar-lo és una sola instrucció que corre en codi C compilat, sense
  passar per l'intèrpret de Python element a element.

A canvi d'aquesta velocitat, un array és menys flexible que una llista: tots els seus
elements han de ser del mateix tipus. Per a taules de números, és exactament el que
voleu.

## 2. L'array: crear-lo i mirar-lo

La manera més directa de crear un array és a partir d'una llista.

In [ ]:
mesures = np.array([5.1, 3.5, 1.4, 0.2])
print(mesures)
print(type(mesures))

Un array té propietats que una llista no té, i que fareu servir constantment:

- **`shape`**: la forma, quantes files i columnes (o quants elements, si és d'una sola
  dimensió).
- **`dtype`**: el tipus de dada que guarda (`int64`, `float64`...).
- **`ndim`**: en quantes dimensions està organitzat.
- **`size`**: el nombre total d'elements.

In [ ]:
print("shape:", mesures.shape)
print("dtype:", mesures.dtype)
print("ndim: ", mesures.ndim)
print("size: ", mesures.size)

També hi ha funcions per crear arrays sense escriure els números un a un:

- `np.zeros(n)`: `n` zeros.
- `np.ones(n)`: `n` uns.
- `np.arange(inici, final, pas)`: com el `range` de Python, però retorna un array.
- `np.linspace(inici, final, quants)`: `quants` números repartits uniformement entre
  `inici` i `final`, tots dos inclosos.

In [ ]:
print(np.zeros(4))
print(np.ones(3))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))

### Arrays de dues dimensions: taules

Fins ara els arrays tenien una sola fila. Un array de **dues dimensions** és una
taula: files i columnes. I aquí ve la connexió amb el que ja coneixeu: **un dataset
és exactament això**. Cada fila és una mostra (una flor), cada columna és una
característica (una mesura).

Ho comprovem amb el dataset d'iris, que ja heu vist.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data

print("shape:", X.shape)
print("dtype:", X.dtype)
print("ndim: ", X.ndim)
print()
print("Les 3 primeres files:")
print(X[:3])

`X` té 150 files (150 flors) i 4 columnes (les 4 mesures: llargada i amplada del
sèpal, llargada i amplada del pètal). Tot el dataset és **un sol array**, res de 150
variables soltes. Aquesta és la forma en què treballareu sempre: files = mostres,
columnes = característiques.

## 3. Indexació i llesques

Amb un array de dues dimensions podeu agafar un element, una fila sencera, una
columna sencera o un tros, amb la notació `array[files, columnes]`.

In [ ]:
print("Element (fila 0, columna 2):", X[0, 2])
print()
print("Fila 0 sencera (una flor):", X[0])
print()
print("Columna 0 sencera, 5 primers valors:", X[:5, 0])
print()
print("Un tros: files 10 a 13, columnes 1 a 3:")
print(X[10:13, 1:3])

La construcció més important de totes és `X[:, columna]`: **totes les files, una
columna**. `:` sol vol dir «tot en aquest eix». Això és exactament com s'agafa una
sola característica de tot el dataset, per exemple totes les llargades de pètal.

In [ ]:
petal_llarg = X[:, 2]
print("shape:", petal_llarg.shape)
print(petal_llarg[:10])

### El parany: una llesca no és una còpia

Quan agafeu un tros d'un array amb `:`, **no es crea un array nou**: és una finestra
sobre la mateixa memòria. Si el modifiqueu, modifiqueu l'original. És una font d'errors
clàssica, perquè amb llistes de Python no passa el mateix amb `.copy()` explícit... i
amb arrays tampoc us avisa ningú.

In [ ]:
prova = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
tros = prova[1:3]
tros[0] = 999.0

print("tros:  ", tros)
print("prova: ", prova)   # el 999 també hi és aquí

Si el que voleu és treballar amb una còpia independent, ho heu de demanar exprés
amb `.copy()`.

In [ ]:
prova2 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
tros_copia = prova2[1:3].copy()
tros_copia[0] = 999.0

print("tros_copia:", tros_copia)
print("prova2:    ", prova2)   # aquesta vegada no ha canviat

## 4. Operacions vectoritzades

Amb NumPy, sumar, multiplicar o elevar al quadrat un array sencer és una sola
operació, sense bucle.

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([10.0, 20.0, 30.0, 40.0])

print("a + b:  ", a + b)
print("a * b:  ", a * b)
print("a ** 2: ", a ** 2)
print("b / a:  ", b / a)

Comparem-ho amb el bucle equivalent, que és el que hauríeu d'escriure sense NumPy.

In [ ]:
suma_bucle = []
for i in range(len(a)):
    suma_bucle.append(a[i] + b[i])

print("Amb bucle:  ", suma_bucle)
print("Amb NumPy:  ", a + b)

Mateix resultat, però `a + b` és més curt, més clar, i molt més ràpid quan l'array
té milers de files en lloc de 4.

NumPy porta també versions vectoritzades de les funcions matemàtiques habituals:
`np.sqrt` (arrel quadrada), `np.exp` (exponencial), `np.abs` (valor absolut)...

In [ ]:
valors = np.array([4.0, -9.0, 16.0, -25.0])

print("np.sqrt(np.abs(valors)):", np.sqrt(np.abs(valors)))
print("np.exp(np.array([0, 1, 2])):", np.exp(np.array([0, 1, 2])))

## 5. Màscares booleanes

Comparar un array amb un número no dona un sol `True`/`False`: dona un **array de
`True`/`False`**, un per element. A això se'n diu una **màscara booleana**.

In [ ]:
petal_llarg = X[:, 2]
mascara = petal_llarg > 5

print(mascara[:10])
print("Quantes flors tenen el pètal > 5 cm:", mascara.sum())

`mascara.sum()` funciona perquè `True` val `1` i `False` val `0`. I aquí ve el
truc que fareu servir constantment: si useu la màscara com a índex, **us selecciona
només les files on és `True`**.

In [ ]:
flors_pètal_llarg = X[mascara]

print("Files seleccionades:", flors_pètal_llarg.shape)
print(flors_pètal_llarg[:5])

Amb una sola línia: agafeu totes les flors amb pètal llarg, sense escriure cap
bucle ni cap `if`.

Per combinar condicions, **no es pot fer servir `and` / `or`**, perquè aquestes
paraules de Python esperen un sol booleà, no un array sencer. Es fan servir els
operadors `&` (i) i `|` (o), i cada condició s'ha de posar entre parèntesis.

In [ ]:
petal_ample = X[:, 3]

mascara_combinada = (petal_llarg > 4) & (petal_ample > 1.5)
print("Flors amb pètal llarg > 4 I ample > 1.5:", mascara_combinada.sum())

mascara_o = (petal_llarg > 6) | (petal_ample > 2.4)
print("Flors amb pètal llarg > 6 O ample > 2.4:", mascara_o.sum())

Finalment, `np.where(condició, valor_si_true, valor_si_false)` és una manera
d'aplicar aquest «si/sinó» element a element, sense escollir files: transforma cada
valor segons la condició.

In [ ]:
etiqueta = np.where(petal_llarg > 5, "pètal llarg", "pètal curt")
print(etiqueta[:10])

## 6. Agregacions i eixos

Les agregacions resumeixen un array en un sol número (o en uns quants): `sum`,
`mean`, `std` (desviació típica), `min`, `max`, i també `argmin`/`argmax`, que no
donen el valor mínim/màxim sinó **la posició** on hi és.

In [ ]:
print("Mitjana del pètal (llargada):", petal_llarg.mean())
print("Desviació típica:        ", petal_llarg.std())
print("Mínim:                    ", petal_llarg.min())
print("Màxim:                    ", petal_llarg.max())
print("Posició del màxim:        ", petal_llarg.argmax())

Fins ara hem agregat un array d'una sola columna. Amb `X`, que té 150 files i 4
columnes, cal dir **en quina direcció** es fa l'agregació. Això és l'`axis`, i és el
concepte que més costa d'entendre.

Mireu-ho amb una taula petita, de 3 files i 2 columnes:

In [ ]:
petita = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
])

print("Suma total (sense axis):     ", petita.sum())
print("Suma per columna (axis=0):   ", petita.sum(axis=0))
print("Suma per fila (axis=1):      ", petita.sum(axis=1))

La regla:

- **`axis=0`** recorre **les files** (de dalt a baix) i us dona **un valor per
  columna**. Amb la taula petita: `1+2+3=6` a la primera columna, `10+20+30=60` a la
  segona → `[6, 60]`.
- **`axis=1`** recorre **les columnes** (d'esquerra a dreta) i us dona **un valor per
  fila**. `1+10=11`, `2+20=22`, `3+30=33` → `[11, 22, 33]`.

Un truc per no confondre-us mai més: `axis=0` **s'esborra la dimensió de les files**
(us queda un valor per columna), i `axis=1` **s'esborra la dimensió de les
columnes** (us queda un valor per fila).

Amb `X` (150 flors × 4 mesures), gairebé sempre voldreu `axis=0`: una mitjana, un
mínim o un màxim **per característica**, mirant totes les flors.

In [ ]:
print("Mitjana de cada columna (axis=0):", X.mean(axis=0))
print("Mínim de cada columna (axis=0):  ", X.min(axis=0))
print("Màxim de cada columna (axis=0):  ", X.max(axis=0))

Quatre números, un per mesura: llargada del sèpal, amplada del sèpal, llargada del
pètal, amplada del pètal. Si haguéssiu fet `axis=1`, hauríeu obtingut 150 números
(la mitjana de les 4 mesures de cada flor), que aquí no serveix de res: no té sentit
mitjanar centímetres de coses diferents dins d'una mateixa flor.

## 7. Dues coses que ja heu vist sense saber-ho

Amb el que porteu ja podeu escriure les dues fórmules més importants del curs.

### La distància euclidiana

Quan vau veure com un model decideix a quina espècie pertany una flor mirant quines
li «s'assemblen» més, la manera de mesurar «assemblar-se» és la distància
euclidiana. Amb NumPy és una línia: la resta, el quadrat, la suma i l'arrel.

In [ ]:
flor_a = X[0]     # una setosa
flor_b = X[50]    # una versicolor

distancia = np.sqrt(((flor_a - flor_b) ** 2).sum())
print("Flor A:", flor_a)
print("Flor B:", flor_b)
print(f"Distància entre A i B: {distancia:.2f}")

Això és exactament el cor del model dels **k veïns més propers (k-NN)**: calcular
aquesta distància entre la flor que voleu classificar i totes les altres, i quedar-vos
amb les més properes.

### La precisió d'un model

Quan un model fa prediccions, saber quantes ha encertat és comparar dos arrays i fer
una mitjana d'una màscara booleana.

In [ ]:
np.random.seed(42)
reals = iris.target
# simulem un model imperfecte que a vegades s'equivoca, només per veure el càlcul
prediccions = reals.copy()
errades = np.random.choice(len(reals), size=8, replace=False)
prediccions[errades] = (prediccions[errades] + 1) % 3

encerts = (prediccions == reals)
print(encerts[:15])
print(f"Precisió: {encerts.mean():.1%}")

`prediccions == reals` és una màscara booleana (`True` on ha encertat), i la seva
mitjana és, directament, la fracció d'encerts. Cada vegada que un model us doni una
«precisió del 95 %», per sota hi ha exactament aquest càlcul.

## 8. Broadcasting

Què passa quan sumeu un sol número a un array sencer? NumPy no us obliga a repetir
el número: l'aplica a tots els elements. A això se'n diu **broadcasting**.

In [ ]:
print(petal_llarg[:5])
print(petal_llarg[:5] + 1)     # suma 1 a cada element
print(petal_llarg[:5] * 2)     # multiplica cada element per 2

El mateix passa quan combineu una **matriu** (150×4) amb un **vector** d'una sola
fila (4 valors): NumPy repeteix el vector per a cada fila de la matriu, sense que
calgui escriure-ho vosaltres.

No calen les regles completes del broadcasting per al curs. Us n'hi ha prou amb el
cas pràctic que fareu servir sempre: **escalar les dades**, restant la mitjana de cada
columna i dividint per la seva desviació típica. És exactament el que fa
`StandardScaler` de scikit-learn per sota.

In [ ]:
mitjana = X.mean(axis=0)   # 1 valor per columna: (4,)
desviacio = X.std(axis=0)  # 1 valor per columna: (4,)

print("Mitjana de cada columna:  ", mitjana)
print("Desviació de cada columna:", desviacio)

X_escalat = (X - mitjana) / desviacio

print()
print("Abans d'escalar, primera flor: ", X[0])
print("Després d'escalar, primera flor:", X_escalat[0])
print()
print("Mitjana de cada columna un cop escalat:", X_escalat.mean(axis=0).round(2))
print("Desviació de cada columna un cop escalat:", X_escalat.std(axis=0).round(2))

`X` té forma `(150, 4)` i `mitjana` té forma `(4,)`: no fan falta bucles ni repetir
el vector 150 vegades a mà, NumPy ho fa sol perquè les mides «encaixen» (4 columnes a
banda i banda). Un cop escalades, totes les columnes tenen mitjana 0 i desviació
típica 1: queden comparables encara que originalment estiguessin en escales molt
diferents.

## 9. Pràctica

Els exercicis fan servir `iris.data` (l'array `X`, amb 150 files i 4 columnes:
llargada sèpal, amplada sèpal, llargada pètal, amplada pètal) i `iris.target`
(l'espècie de cada flor, com a número 0/1/2).

### Exercici 1 — La flor amb el pètal més llarg

Troba la **posició** (l'índex de fila) de la flor amb la llargada de pètal més gran
de tot el dataset, i imprimeix les seves 4 mesures.

Pista: la llargada del pètal és la columna 2. El mètode `.argmax()` et dona
directament la **posició** del màxim, no el valor; el valor el dona `.max()`.

In [ ]:
# Exercici 1

# 1. queda't amb la columna 2 (llargada del pètal) amb X[:, 2]
# 2. troba la posició del valor més gran amb .argmax()
# 3. imprimeix les 4 mesures de la flor en aquesta posició (X[posicio])

**Com saps que ho has fet bé:** el valor de la columna 2 en aquesta posició ha de
ser igual a `petal_llarg.max()`.

### Exercici 2 — La mitjana, a mà i amb `mean`

Calcula la mitjana de cada una de les 4 columnes de `X` de dues maneres: primer amb
un bucle que sumi els valors i divideixi pel nombre de files, i després amb
`X.mean(axis=0)`. Compara els dos resultats.

In [ ]:
# Exercici 2

n_files, n_columnes = X.shape
mitjanes_bucle = []

# per cada columna j: agafa-la amb X[:, j], suma-la amb un bucle i divideix per n_files
# desa cada resultat a mitjanes_bucle amb .append()

mitjanes_numpy = X.mean(axis=0)

print("Amb bucle:", mitjanes_bucle)
print("Amb mean: ", mitjanes_numpy)

**Com saps que ho has fet bé:** els 4 números de `mitjanes_bucle` han de coincidir
(fins als decimals) amb els de `mitjanes_numpy`.

### Exercici 3 — Comptar amb una màscara

Compta quantes flors tenen la llargada de sèpal (columna 0) per sobre de 6 cm, i
quin percentatge representen sobre el total de 150.

In [ ]:
# Exercici 3

# crea la màscara booleana comparant X[:, 0] amb 6 (operador >)
# compta quants True té amb .sum() i calcula el percentatge sobre len(X)

**Com saps que ho has fet bé:** si sumes els que tenen sèpal > 6 i els que tenen
sèpal <= 6, has de tornar a obtenir 150.

### Exercici 4 — Normalitza una columna

Agafa la columna de la llargada del pètal (columna 2) i normalitza-la: resta-li la
seva mitjana i divideix pel resultat entre la seva desviació típica. Comprova que la
columna resultant té mitjana pràcticament 0 i desviació típica pràcticament 1.

In [ ]:
# Exercici 4

petal_llarg = X[:, 2]

# calcula la mitjana amb .mean() i la desviació típica amb .std() de petal_llarg
# normalitza: (petal_llarg - mitjana) / desviacio
# imprimeix .mean() i .std() del resultat

**Com saps que ho has fet bé:** la mitjana ha de sortir molt a prop de `0` (per
exemple `1e-16`, que és pràcticament 0) i la desviació típica molt a prop de `1`.

### Exercici 5 — La flor més propera (k-NN amb k=1)

Agafa la flor de la posició 0. Calcula la seva distància euclidiana a **totes** les
altres 149 flors (sense comptar-se a ella mateixa) i troba quina és la més propera.
Imprimeix la seva posició i l'espècie de totes dues (`iris.target[posicio]`).

Pista: pots calcular totes les distàncies d'un sol cop amb broadcasting:
`np.sqrt(((X - flor_0) ** 2).sum(axis=1))` et dona un array amb 150 distàncies, una
per fila. Per no triar la mateixa flor 0 (distància 0), posa-li manualment un valor
molt gran abans de buscar el mínim.

In [ ]:
# Exercici 5

flor_0 = X[0]

# calcula distancies amb la fórmula de la pista: np.sqrt, ** 2 i .sum(axis=1)
# posa distancies[0] a un valor molt gran (per exemple np.inf) perquè no es triï a si mateixa
# troba la posició de la distància mínima amb .argmin()

# imprimeix la posició trobada, i les espècies (iris.target) de la flor 0 i de la flor trobada

**Com saps que ho has fet bé:** `distancies[0]` ha de valer `0.0` abans de
substituir-la, i la posició que trobis no pot ser `0`.

### Exercici 6 — Combinar màscares

Compta quantes flors compleixen alhora que el pètal és llarg (columna 2 > 4.5) **i**
el sèpal és ample (columna 1 > 3). Fes-ho també per separat (quantes compleixen
només la primera condició, quantes només la segona) i compara amb el resultat
combinat.

In [ ]:
# Exercici 6

# mascara_petal = X[:, 2] > 4.5
# mascara_sepal = X[:, 1] > 3
# combina-les amb & i compta amb .sum()
# compta també cada màscara per separat amb .sum() i compara les tres xifres

**Com saps que ho has fet bé:** el nombre de flors que compleixen totes dues
condicions no pot ser mai més gran que el nombre que compleix qualsevol de les dues
per separat.

### Exercici 7 — Mitjana per espècie amb una màscara

`iris.target` val `0` per a les setosa, `1` per a les versicolor i `2` per a les
virginica. Calcula la mitjana de la llargada del pètal (columna 2) **només** de les
flors virginica (`iris.target == 2`), sense fer servir cap bucle.

In [ ]:
# Exercici 7

# crea la màscara iris.target == 2
# aplica-la a la columna 2 de X (X[mascara, 2] o X[:, 2][mascara])
# calcula la mitjana del resultat amb .mean()

**Com saps que ho has fet bé:** aquesta mitjana ha de ser clarament més gran que
`petal_llarg.mean()` (la mitjana de totes les espècies juntes), perquè la virginica
és l'espècie amb el pètal més llarg.

## Resum

- Un array de NumPy és més ràpid que una llista de Python perquè guarda **un sol
  tipus** de dada en **memòria contiguïa**, i opera amb codi vectoritzat en lloc
  d'un bucle element a element.
- Un dataset és un array de 2 dimensions: **files = mostres, columnes =
  característiques**. `X[:, columna]` és com s'agafa una característica sencera.
- Una llesca no és una còpia: si la modifiqueu, modifiqueu l'original. Feu servir
  `.copy()` quan calgui una còpia independent.
- Les **operacions vectoritzades** (`+`, `*`, `**`, `np.sqrt`...) treballen sobre
  tot l'array sense bucles.
- Les **màscares booleanes** (`X > valor`) permeten seleccionar i comptar files que
  compleixen una condició. Per combinar-les, `&` i `|`, mai `and`/`or`.
- `axis=0` agrega **per columna** (recorrent les files), `axis=1` agrega **per fila**
  (recorrent les columnes).
- Ja sabeu escriure les dues fórmules que fareu servir constantment: la **distància
  euclidiana** (`np.sqrt(((a - b) ** 2).sum())`, el cor de k-NN) i la **precisió**
  d'un model (`(prediccions == reals).mean()`).
- El **broadcasting** us permet combinar un array amb un vector més petit (per
  exemple, restar la mitjana de cada columna a tota la taula) sense escriure cap
  bucle. És exactament el que fa `StandardScaler`.

### I ara què

Amb NumPy ja podeu manejar les dades que necessita qualsevol model. El proper pas és
fer servir de veritat el que heu vist conceptualment amb Iris: implementar i entrenar
els vostres primers models de machine learning.